In [1]:
%load_ext autoreload
%autoreload 2

### Setup

In [2]:
import os, sys
sys.path.append(os.path.dirname(os.getcwd())) # Add the parent directory to sys.path

import json
import pandas as pd
import numpy as np

from utils import DATA_DIR
from download.weka import pull_predictions_from_weka

from old.dataloader_legacy import process_instance_stats_df, get_mix_nd_array, compute_significance, get_slice, get_mix_nd_array

In [3]:
pull_predictions_from_weka("consistent_ranking") # preprocessing takes ~80 min

Downloading: 100%|█████████████████████████| 5.80G/5.80G [13:47<00:00, 7.01MB/s]


#### Across checkpoints

In [4]:
COLS = ['step', 'model', 'task', 'mix', 'size', 'token_ratio', 'native_id', 'acc_per_char'] # load a subset of columns to save on memory

df = pd.read_parquet(f'{DATA_DIR}/all_consistent_ranking_predictions.parquet', columns=COLS)

# import pyarrow.parquet as pq
# table = pq.read_table(f'{DATA_DIR}/all_consistent_ranking_predictions.parquet', columns=['step', 'model', 'task', 'mix', 'native_id', 'acc_per_char'])
# print(table)

In [5]:
TASKS = df.index.get_level_values('task').unique().to_list()

In [6]:
mixes, scores = get_mix_nd_array(df, '1B', 'arc_easy', 'acc_per_char')

KeyError: 'step'

In [ ]:
steps = df.index.get_level_values('step').unique().sort_values()

scores = []
for step in steps:
    mixes, score = get_mix_nd_array(df, '1B', 'arc_easy', 'acc_per_char', step=step, sorted=False)
    if score.size > 0: scores += [score]

scores_ckpt = np.stack(scores)

#### Across sizes

In [ ]:
import json
import pandas as pd
import numpy as np

with open('instance_stats_mixes_para.json', 'r', encoding='utf-8') as f: instance_stats = json.load(f)
with open('instance_stats_mixes_rc.json', 'r', encoding='utf-8') as f: 
    new_stats = json.load(f)
    for k in instance_stats: instance_stats[k].update(new_stats[k])
    
instance_stats_df = pd.DataFrame(instance_stats)
instance_df = process_instance_stats_df(instance_stats_df)

In [ ]:
mixes, scores = get_mix_nd_array(instance_df, '1B', 'arc_easy', 'acc_per_char')

In [ ]:
sizes = ['150M', '300M', '530M', '750M', '1B']

scores_scale = []
for size in sizes:
    mixes, score = get_mix_nd_array(instance_df, size, 'arc_easy', 'acc_per_char', sorted=False)
    scores_scale += [score]

scores_scale = np.stack(scores_scale)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# n_flips = np.abs(np.diff(scores_scale, axis=0)).sum(axis=0).mean(axis=0)
n_flips_ckpt = np.abs(np.diff(scores_ckpt, axis=0)).sum(axis=0).mean(axis=0)
avg_accuracy_scale = np.mean(scores_scale[:, :, :], axis=(0, 1))
avg_accuracy_ckpt = np.mean(scores_ckpt[:, :, :], axis=(0, 1))

# Plot mean vs variance
scatter = plt.scatter(n_flips_ckpt, avg_accuracy_scale, c=avg_accuracy_ckpt, s=1, alpha=0.5)
plt.xlabel('# flips across model scales (avg. across mixes)')
plt.ylabel('avg. accuracy across scale (avg. across mixes)')
cbar = plt.colorbar(scatter)
cbar.set_label('avg. accuracy across checkpoints at 1B (avg. across mixes)')
plt.title('ARC-e')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# the above graph should have some measure of "entropy", or the number of mixes which agreed on the same answer
# more difficult questions should have less agreement, not high agreement on an incorrect answer

n_flips_ckpt = np.abs(np.diff(scores_ckpt, axis=0)).sum(axis=0).mean(axis=0)

_, target_scores = get_mix_nd_array(instance_df_ckpt, '1B', 'arc_easy', 'acc_per_char', sorted=False)
_, answers = get_mix_nd_array(instance_df_ckpt, '1B', 'arc_easy', 'predicted_index_raw', sorted=False)
_, correct = get_mix_nd_array(instance_df_ckpt, '1B', 'arc_easy', 'correct_choice', sorted=False)

def compute_entropy(array, axis=None):
    if axis is None: array = array.flatten()
    _, counts = np.unique(array, return_counts=True, axis=axis)
    probabilities = counts / counts.sum()
    entropy = -np.sum(probabilities * np.log(probabilities), axis=-1)
    return entropy
entropy = np.apply_along_axis(compute_entropy, axis=1, arr=answers.T)

target_scores = target_scores.mean(axis=0)
target_scores_2 = (answers == correct).sum(axis=0)

# Plot
scatter = plt.scatter(entropy, target_scores_2, c=n_flips_ckpt, s=1, alpha=0.5)
plt.xlabel('entropy across model choices')
plt.ylabel('# correct models')
cbar = plt.colorbar(scatter)
cbar.set_label('# flips across model scale')
plt.title('ARC-e (20 mixes, 1B final checkpoint)')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

diff = np.abs(np.diff(scores, axis=0)).sum(axis=0).mean(axis=0)
scales = diff
mixes = np.mean(scores[:, :, :], axis=(0, 1)) # mean of all mixes, 1B scale

diff_ckpt = np.abs(np.diff(scores_ckpt, axis=0)).sum(axis=0).mean(axis=0)
ckpt = np.mean(scores_ckpt[:, :, :], axis=(0, 1)) # mean of all mixes, all checkpoints

# Plot mean vs variance
scatter = plt.scatter(mixes, ckpt, c=diff_ckpt, s=1, alpha=0.5)
plt.xlabel('avg. accuracy across checkpoints (avg. across mixes)')
plt.ylabel('avg. accuracy across scale (avg. across mixes)')
cbar = plt.colorbar(scatter)
cbar.set_label('# flips across checkpoints (avg. across mixes)')
plt.title('ARC-e')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

diff = np.abs(np.diff(scores, axis=0)).sum(axis=0).mean(axis=0)
scales = diff
# mixes = np.mean(scores[:, :, :], axis=(0, 1)) # mean of all mixes, 1B scale
mixes = np.mean(scores[-1, :, :], axis=0) # mean of all mixes, 1B scale

diff_ckpt = np.abs(np.diff(scores_ckpt, axis=0)).sum(axis=0).mean(axis=0)
# ckpt = np.mean(scores_ckpt[:, :, :], axis=(0, 1)) # mean of all mixes, final checkpoint
ckpt = np.mean(scores_ckpt[-1, :, :], axis=0) # mean of all mixes, final checkpoint

# Plot mean vs variance
scatter = plt.scatter(diff_ckpt, ckpt, c=diff, s=1, alpha=0.5)
plt.xlabel('# flips across checkpoints (avg. across mixes)')
plt.ylabel('avg. accuracy across mixes (1B final checkpoint)')
cbar = plt.colorbar(scatter)
cbar.set_label('# flips across scale (avg. across mixes)')
plt.title('ARC-e')
plt.show()

### Bringing it Together

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# tasks = instance_df_ckpt.index.get_level_values('task_suite').unique().sort_values()
tasks = ['arc_easy', 'arc_easy:para']
steps = instance_df_ckpt.index.get_level_values('step').unique().sort_values()

fig, axes = plt.subplots(len(tasks) // 2, 2, figsize=(10, 3.5*(len(tasks) // 2)), sharey=True)
axes = axes.flatten()

for i, task in tqdm(enumerate(tasks), desc="Plotting tasks", total=len(tasks)):
    scores_ckpt = []
    for step in steps:
        mixes, score = get_mix_nd_array(instance_df_ckpt, '1B', task, 'acc_per_char', step=step)
        if score.size > 0:
            scores_ckpt.append(score)

    scores_ckpt = np.stack(scores_ckpt)

    sizes = ['150M', '300M', '530M', '750M', '1B']

    scores = []
    for size in sizes:
        mixes, score = get_mix_nd_array(instance_df, size, task, 'acc_per_char')
        scores.append(score)

    scores = np.stack(scores)

    mixes = np.mean(scores[:, :, :], axis=(0, 1))  # Mean of all mixes, 1B scale
    diff_ckpt = np.abs(np.diff(scores_ckpt, axis=0)).sum(axis=0).mean(axis=0)
    ckpt = np.mean(scores_ckpt[:, :, :], axis=(0, 1))  # Mean of all mixes, all checkpoints

    ax: plt.Axes = axes[i]
    scatter = ax.scatter(diff_ckpt, mixes, c=ckpt, s=1, alpha=0.5)
    ax.set_title(task)
    ax.set_xlabel('# flips across model scales')
    if i == 0: ax.set_ylabel('avg. accuracy across scale')

fig.colorbar(scatter, ax=axes, orientation='vertical', label='avg. accuracy across checkpoints at 1B')

# plt.suptitle('ARC-e across Tasks')
# plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()